# Marketing Channel Performance: EDA and Statistical Testing

This notebook evaluates campaign performance across Google Ads, Meta Ads, and TikTok Ads. It covers data quality, aggregate marketing metrics, visual exploration, pairwise tests, multiple-comparison control, power analysis, and an executive recommendation.

## 1. Dataset discovery and choice

- **Dataset:** Global Ads Performance (Google, Meta, TikTok)
- **Source:** [Kaggle](https://www.kaggle.com/datasets/nudratabbas/global-ads-performance-google-meta-tiktok/data)
- **Why this dataset:** It contains campaign-day observations for three comparable advertising platforms, with impressions, clicks, spend, conversions, and revenue. That supports both continuous metric comparisons and conversion-rate tests.
- **Unit of observation:** One row is a daily advertising campaign record. The analysis therefore treats rows as campaign-day observations and reports that assumption as a limitation.
- **Scope note:** The requested business scenario mentions seven channels, but this file contains three platforms. The recommendation below is therefore a provisional three-platform allocation, not a seven-channel budget plan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from pathlib import Path
from IPython.display import Markdown, display

from scipy import stats
from scipy.stats import fisher_exact
try:
    from scipy.stats import false_discovery_control
except ImportError:
    def false_discovery_control(pvalues, method='bh'):
        values = np.asarray(pvalues, dtype=float)
        order = np.argsort(values)
        ranked = values[order]
        adjusted = ranked * len(values) / np.arange(1, len(values) + 1)
        adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
        result = np.empty_like(adjusted)
        result[order] = np.minimum(adjusted, 1.0)
        return result

sns.set_theme(style='whitegrid', context='notebook')
ALPHA = 0.05
DATA_PATH = Path('global_ads_performance_dataset.csv')
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

In [ ]:
# Load the local copy of the Kaggle dataset and standardise the two names used below.
COLUMN_MAP = {'platform': 'channel', 'ad_spend': 'cost'}
raw = pd.read_csv(DATA_PATH).rename(columns=COLUMN_MAP)
required = ['date', 'channel', 'impressions', 'clicks', 'conversions', 'cost', 'revenue']
missing_required = sorted(set(required) - set(raw.columns))
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

print(f'Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print(f'Channels: {raw.channel.nunique()} -> {sorted(raw.channel.unique())}')
raw.info()
display(raw.head())

In [ ]:
# Convert types, remove duplicate or impossible records, and calculate row-level metrics.
df = raw.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
numeric_cols = ['impressions', 'clicks', 'conversions', 'cost', 'revenue']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

before = len(df)
df = df.drop_duplicates().dropna(subset=required)
invalid = (
    (df[numeric_cols] < 0).any(axis=1)
    | (df.clicks > df.impressions)
    | (df.conversions > df.clicks)
)
df = df.loc[~invalid].copy()

df['ctr'] = df.clicks.div(df.impressions).replace([np.inf, -np.inf], np.nan)
df['conversion_rate'] = df.conversions.div(df.clicks).replace([np.inf, -np.inf], np.nan)
df['cpa'] = df.cost.div(df.conversions).replace([np.inf, -np.inf], np.nan)
df['roas'] = df.revenue.div(df.cost).replace([np.inf, -np.inf], np.nan)
df['profit'] = df.revenue - df.cost
df['profit_margin'] = df.profit.div(df.revenue).replace([np.inf, -np.inf], np.nan)

print(f'Removed {before - len(df)} duplicate or invalid rows; retained {len(df):,}.')
print(f'Date range: {df.date.min().date()} to {df.date.max().date()} ({df.date.nunique()} dates)')
print(f'Rows per channel:\n{df.channel.value_counts().sort_index()}')
print(f'\nMissing values after cleaning:\n{df.isna().sum()[df.isna().sum() > 0]}')
df.to_csv(OUT / 'marketing_data.csv', index=False)
display(df.describe(include='number').T.round(3))

### Transformations made

Dates and core measures were coerced to their appropriate types. Exact duplicate rows, missing required fields, negative measures, clicks greater than impressions, and conversions greater than clicks were excluded. CTR, conversion rate, CPA, ROAS, profit, and profit margin were calculated from the underlying totals rather than relying on the source's precomputed fields. Infinite ratios were converted to missing values. The cleaned file is saved as `outputs/marketing_data.csv`.

In [ ]:
# Aggregate totals are the appropriate basis for channel-level marketing KPIs.
summary = (df.groupby('channel')
    .agg(rows=('channel', 'size'), impressions=('impressions', 'sum'),
         clicks=('clicks', 'sum'), conversions=('conversions', 'sum'),
         cost=('cost', 'sum'), revenue=('revenue', 'sum'))
)
summary['ctr'] = summary.clicks / summary.impressions
summary['conversion_rate'] = summary.conversions / summary.clicks
summary['cpa'] = summary.cost / summary.conversions
summary['roas'] = summary.revenue / summary.cost
summary['profit'] = summary.revenue - summary.cost
summary['profit_margin'] = summary.profit / summary.revenue
summary = summary.sort_values('roas', ascending=False)
summary.to_csv(OUT / 'channel_summary.csv')
display(summary.style.format({'ctr': '{:.2%}', 'conversion_rate': '{:.2%}', 'cpa': '${:,.2f}', 'roas': '{:.2f}x', 'cost': '${:,.0f}', 'revenue': '${:,.0f}', 'profit': '${:,.0f}', 'profit_margin': '{:.2%}'}))

In [ ]:
# Group-level overview: lower CPA is better; higher ROAS, conversion rate, profit, and conversions are better.
plot_specs = [('cpa', 'CPA ($)', 'viridis'), ('roas', 'ROAS (x)', 'magma'),
              ('conversion_rate', 'Conversion rate', 'crest'), ('conversions', 'Conversions', 'Blues'),
              ('profit', 'Profit ($)', 'Greens'), ('cost', 'Spend ($)', 'Oranges')]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (metric, label, palette) in zip(axes.flat, plot_specs):
    values = summary.sort_values(metric, ascending=(metric == 'cpa'))
    sns.barplot(data=values.reset_index(), y='channel', x=metric, ax=ax, hue='channel', legend=False, palette=palette)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('')
    if metric == 'conversion_rate':
        ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    if metric in ['profit', 'cost']:
        ax.xaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter('${x:,.0f}'))
fig.suptitle('Aggregate channel performance', y=1.02, fontsize=16)
fig.tight_layout()
fig.savefig(OUT / 'group_metrics_overview.png', dpi=160, bbox_inches='tight')
plt.show()

# Row-level distributions show variability across campaign-day observations.
distribution_specs = [('cpa', 'CPA ($)'), ('roas', 'ROAS (x)'), ('conversion_rate', 'Conversion rate')]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for col, (metric, label) in enumerate(distribution_specs):
    clean_metric = df[['channel', metric]].dropna()
    sns.histplot(data=clean_metric, x=metric, hue='channel', element='step', stat='density',
                 common_norm=False, alpha=0.25, ax=axes[0, col])
    axes[0, col].set_title(f'{label} distribution')
    axes[0, col].set_xlabel(label)
    sns.boxplot(data=clean_metric, x='channel', y=metric, hue='channel', legend=False, ax=axes[1, col])
    axes[1, col].set_title(f'{label} by channel')
    axes[1, col].set_xlabel('')
    axes[1, col].set_ylabel(label)
    if metric == 'conversion_rate':
        axes[0, col].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
        axes[1, col].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
fig.tight_layout()
fig.savefig(OUT / 'group_distributions.png', dpi=160, bbox_inches='tight')
plt.show()

## 2. Pairwise comparisons

Welch's independent-samples t-test is used for row-level CPA, ROAS, and conversion rate because the channel sample sizes and variances differ. The reported difference is **group B minus group A**. Cohen's *d* uses the requested pooled standard deviation, `sqrt((var_A + var_B) / 2)`. P-values are retained for correction across all pairwise comparisons within each metric family.

In [ ]:
def cohens_d(a, b):
    pooled_sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return np.nan if pooled_sd == 0 else (b.mean() - a.mean()) / pooled_sd

def effect_label(d):
    magnitude = abs(d)
    if magnitude < 0.2:
        return 'negligible'
    if magnitude < 0.5:
        return 'small'
    if magnitude < 0.8:
        return 'medium'
    return 'large'

def pairwise_t_tests(data, metric):
    results = []
    for group_a, group_b in combinations(sorted(data.channel.unique()), 2):
        a = data.loc[data.channel == group_a, ['channel', metric]].dropna()[metric].to_numpy()
        b = data.loc[data.channel == group_b, ['channel', metric]].dropna()[metric].to_numpy()
        test = stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')
        difference = b.mean() - a.mean()
        d = cohens_d(a, b)
        results.append({'metric': metric, 'group_a': group_a, 'group_b': group_b,
                        'n_a': len(a), 'n_b': len(b), 'mean_a': a.mean(), 'mean_b': b.mean(),
                        'difference_b_minus_a': difference,
                        'pct_difference_b_minus_a': np.nan if a.mean() == 0 else 100 * difference / a.mean(),
                        't_statistic': test.statistic, 'p_value': test.pvalue,
                        'cohens_d': d, 'effect_size': effect_label(d),
                        'significant_raw': test.pvalue < ALPHA})
    return pd.DataFrame(results)

t_results = pd.concat([pairwise_t_tests(df, metric) for metric in ['cpa', 'roas', 'conversion_rate']], ignore_index=True)
t_results.to_csv(OUT / 'pairwise_t_tests.csv', index=False)
display(t_results.round({'mean_a': 4, 'mean_b': 4, 'difference_b_minus_a': 4, 'pct_difference_b_minus_a': 2, 't_statistic': 3, 'p_value': 6, 'cohens_d': 3}))
print(f'Raw significant comparisons: {int(t_results.significant_raw.sum())} of {len(t_results)}')

In [ ]:
# P-value heatmap for the primary business metric, ROAS.
channels = sorted(df.channel.unique())
p_matrix = pd.DataFrame(1.0, index=channels, columns=channels)
for row in t_results.query("metric == 'roas'").itertuples():
    p_matrix.loc[row.group_a, row.group_b] = row.p_value
    p_matrix.loc[row.group_b, row.group_a] = row.p_value

plt.figure(figsize=(7, 5))
sns.heatmap(p_matrix, annot=True, fmt='.2g', cmap='RdYlGn_r', vmin=0, vmax=0.05,
            linewidths=0.5, cbar_kws={'label': 'p-value, capped at 0.05'})
plt.title('Pairwise Welch t-test p-values: ROAS')
plt.tight_layout()
plt.savefig(OUT / 'roas_pvalue_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. Binary outcome comparison with Fisher's exact test

For each channel pair, conversions and non-conversions are constructed from total clicks. This tests whether the click-level conversion odds differ. Because the counts are aggregated across campaign-day rows, the test can be extremely sensitive; practical differences in conversion rate are reported alongside the p-value.

In [ ]:
fisher_rows = []
for group_a, group_b in combinations(channels, 2):
    a = summary.loc[group_a]
    b = summary.loc[group_b]
    a_conv, a_non = int(a.conversions), int(a.clicks - a.conversions)
    b_conv, b_non = int(b.conversions), int(b.clicks - b.conversions)
    # Put group B first so the odds ratio is interpreted as B relative to A.
    odds_ratio, p_value = fisher_exact([[b_conv, b_non], [a_conv, a_non]], alternative='two-sided')
    fisher_rows.append({'group_a': group_a, 'group_b': group_b, 'clicks_a': int(a.clicks),
                       'clicks_b': int(b.clicks), 'conversions_a': a_conv, 'conversions_b': b_conv,
                       'conversion_rate_a': a.conversions / a.clicks,
                       'conversion_rate_b': b.conversions / b.clicks,
                       'rate_difference_b_minus_a': b.conversions / b.clicks - a.conversions / a.clicks,
                       'odds_ratio_b_vs_a': odds_ratio, 'p_value': p_value,
                       'significant_raw': p_value < ALPHA})
fisher_results = pd.DataFrame(fisher_rows)
fisher_results.to_csv(OUT / 'fisher_exact_tests.csv', index=False)
display(fisher_results.round({'conversion_rate_a': 4, 'conversion_rate_b': 4, 'rate_difference_b_minus_a': 4, 'odds_ratio_b_vs_a': 4, 'p_value': 6}))

## 4. Multiple-comparison correction

Both Bonferroni and Benjamini–Hochberg (BH) adjustments are applied within each test family: each continuous metric's pairwise t-tests and the Fisher tests. Bonferroni controls family-wise error more conservatively; BH controls the expected false discovery rate.

In [ ]:
def add_corrections(results):
    results = results.copy()
    p = results.p_value.to_numpy()
    results['p_bonferroni'] = np.minimum(p * len(p), 1.0)
    results['p_bh'] = false_discovery_control(p, method='bh')
    results['significant_bonferroni'] = results.p_bonferroni < ALPHA
    results['significant_bh'] = results.p_bh < ALPHA
    return results

corrected_t = t_results.groupby('metric', group_keys=False).apply(add_corrections, include_groups=False).reset_index(drop=True)
corrected_fisher = add_corrections(fisher_results)
corrected_t.to_csv(OUT / 'pairwise_t_tests_corrected.csv', index=False)
corrected_fisher.to_csv(OUT / 'fisher_exact_tests_corrected.csv', index=False)

correction_summary = []
for name, result in [('CPA t-tests', corrected_t.query("metric == 'cpa'")),
                     ('ROAS t-tests', corrected_t.query("metric == 'roas'")),
                     ('Conversion-rate t-tests', corrected_t.query("metric == 'conversion_rate'")),
                     ('Fisher exact tests', corrected_fisher)]:
    correction_summary.append({'test_family': name, 'comparisons': len(result),
                              'raw_significant': int(result.significant_raw.sum()),
                              'bonferroni_significant': int(result.significant_bonferroni.sum()),
                              'bh_significant': int(result.significant_bh.sum())})
display(pd.DataFrame(correction_summary))

## 5. Power analysis

The following calculation uses the noncentral t distribution for a two-sided independent-samples test with equal future group sizes. It estimates the observations per group needed for 80% power at alpha = 0.05 for small, medium, and large Cohen's *d*. It also calculates observed power for the current pairwise ROAS comparisons.

In [ ]:
def t_test_power(effect_size, n_a, n_b=None, alpha=ALPHA):
    n_b = n_a if n_b is None else n_b
    dfree = n_a + n_b - 2
    ncp = abs(effect_size) * np.sqrt(n_a * n_b / (n_a + n_b))
    critical = stats.t.ppf(1 - alpha / 2, dfree)
    return stats.nct.sf(critical, dfree, ncp) + stats.nct.cdf(-critical, dfree, ncp)

def required_n_per_group(effect_size, target_power=0.80, alpha=ALPHA, max_n=10000):
    for n in range(2, max_n + 1):
        if t_test_power(effect_size, n) >= target_power:
            return n
    return np.nan

power_plan = pd.DataFrame({'cohens_d': [0.2, 0.5, 0.8]})
power_plan['interpretation'] = ['small', 'medium', 'large']
power_plan['required_n_per_group_for_80pct_power'] = power_plan.cohens_d.map(required_n_per_group)
power_plan.to_csv(OUT / 'power_sample_size_plan.csv', index=False)
display(power_plan)

roas_results = corrected_t.query("metric == 'roas'").copy()
roas_results['observed_power'] = roas_results.apply(lambda r: t_test_power(r.cohens_d, int(r.n_a), int(r.n_b)), axis=1)
display(roas_results[['group_a', 'group_b', 'n_a', 'n_b', 'cohens_d', 'observed_power']].round(3))

min_n = int(df.channel.value_counts().min())
medium_n = int(power_plan.loc[power_plan.cohens_d == 0.5, 'required_n_per_group_for_80pct_power'].iloc[0])
display(Markdown(f'**Planning implication:** the smallest current channel has {min_n:,} campaign-day observations, compared with {medium_n} per group for 80% power to detect a medium effect. Small effects require substantially more observations, and this calculation does not correct for repeated measurements or channel confounding.'))

## 6. Executive memo and recommendation

In [ ]:
best_roas = summary.roas.idxmax()
best_cpa = summary.cpa.idxmin()
roas_all_bh = bool(roas_results.significant_bh.all())
conversion_t_tests = corrected_t.query("metric == 'conversion_rate'")
conversion_t_bh_count = int(conversion_t_tests.significant_bh.sum())

# This is a cautious three-platform test allocation, not a claim that spend scales linearly.
allocation = pd.DataFrame({'channel': ['TikTok Ads', 'Meta Ads', 'Google Ads', 'Validation reserve'],
                          'share': [0.45, 0.30, 0.15, 0.10]})
allocation['monthly_amount'] = allocation.share * 500_000
allocation.to_csv(OUT / 'illustrative_500k_allocation.csv', index=False)
display(allocation.style.format({'share': '{:.0%}', 'monthly_amount': '${:,.0f}'}))

memo = f'''# Executive memo: provisional channel budget recommendation

## Finding

Across {len(df):,} campaign-day observations, {best_roas} had the highest aggregate ROAS ({summary.loc[best_roas, 'roas']:.2f}x) and {best_cpa} had the lowest aggregate CPA (${summary.loc[best_cpa, 'cpa']:.2f}). Pairwise ROAS Welch tests remained significant after both Bonferroni and Benjamini–Hochberg correction: {roas_all_bh}. The ROAS effects range from medium to large, so the differences are practically meaningful in this dataset.

Aggregate conversion rates were close ({summary.loc['Google Ads', 'conversion_rate']:.2%} Google, {summary.loc['Meta Ads', 'conversion_rate']:.2%} Meta, and {summary.loc['TikTok Ads', 'conversion_rate']:.2%} TikTok). The pooled-click Fisher tests are statistically significant, while the row-level conversion-rate t-tests have {conversion_t_bh_count} BH-significant pair(s). This contrast suggests that statistical significance is being driven partly by very large click counts; conversion-rate differences should not be treated as large commercial effects without a controlled experiment.

## Provisional $500,000 plan

The supplied data has three platforms rather than seven channels, so the following is an illustrative starting point only:

- TikTok Ads: 45% ($225,000)
- Meta Ads: 30% ($150,000)
- Google Ads: 15% ($75,000)
- Validation reserve: 10% ($50,000)

The reserve should fund randomized holdouts, creative tests, and measurement. Before reallocating the full budget, obtain comparable data for the four missing channels and validate whether the observed ROAS ranking persists after controlling for campaign type, industry, country, seasonality, and spend level.

## Caveats and next steps

These are observational campaign-day comparisons, not causal estimates. Rows may share campaigns, audiences, or dates, and aggregate Fisher tests treat clicks as independent. ROAS and CPA can also be skewed by attribution and outliers. Use the current results to prioritize tests, not to promise incremental returns. The power analysis indicates adequate row counts for medium effects, but small effects need larger or better-designed experiments.'''
display(Markdown(memo))
(OUT / 'executive_memo.md').write_text(memo, encoding='utf-8')
print(f'Wrote analysis artifacts to {OUT.resolve()}')